In [0]:
df = spark.table("samples.bakehouse.sales_customers")
display(df)

In [0]:
# Default - displays 20 rows and 
# 20 charactes from column value 
# df.show()

# #Display full column contents
# df.show(truncate=False)

# # Display 2 rows and full column contents
# df.show(2,truncate=False) 

# # Display 2 rows & column values 25 characters
# df.show(2,truncate=25) 

# # # Display DataFrame rows & columns vertically
df.show(n=3,truncate=25,vertical=True)

In [0]:
# Select columns by different ways
df.select("first_name","last_name").show() # medium
df.select(df.first_name,df.last_name).show() # fast
df.select(df["first_name"],df["last_name"]).show() # slow



In [0]:
#Selects first 3 columns and top 3 rows
df.select(df.columns[:3]).show(3)

#Selects columns 2 to 4  and top 3 rows
df.select(df.columns[2:4]).show(3,truncate=25)

In [0]:
# Create DataFrame with nested columns
data = [
        (("James",None,"Smith"),"OH","M"),
        (("Anna","Rose",""),"NY","F"),
        (("Julia","","Williams"),"OH","F"),
        (("Maria","Anne","Jones"),"NY","M"),
        (("Jen","Mary","Brown"),"NY","M"),
        (("Mike","Mary","Williams"),"OH","M")
        ]

from pyspark.sql.types import StructType,StructField, StringType        
schema = StructType([
    StructField('name', StructType([
         StructField('firstname', StringType(), True),
         StructField('middlename', StringType(), True),
         StructField('lastname', StringType(), True)
         ])),
     StructField('state', StringType(), True),
     StructField('gender', StringType(), True)
     ])
df2 = spark.createDataFrame(data = data, schema = schema)
df2.printSchema()
df2.show(truncate=False) # shows all columns



In [0]:
# Select struct column
df2.select("name").show(truncate=False)



In [0]:
# Select child columns
df2.select("name.firstname","name.lastname").show(truncate=False)

In [0]:
# Select all child columns
df2.select("name.*").show(truncate=False)

In [0]:
data = [('James','','Smith','1991-04-01','M',3000),
  ('Michael','Rose','','2000-05-19','M',4000),
  ('Robert','','Williams','1978-09-05','M',4000),
  ('Maria','Anne','Jones','1967-12-01','F',4000),
  ('Jen','Mary','Brown','1980-02-17','F',-1)
]

columns = ["firstname","middlename","lastname","dob","gender","salary"]
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('SparkByExamples.com').getOrCreate()
df = spark.createDataFrame(data=data, schema = columns)

In [0]:
from pyspark.sql.functions import col

#here every thing is string data type we use withcolumn to cast the  salary as integer
display(df.withColumn("salary", col("salary").cast("Integer")))

In [0]:
df2 = df.withColumn("salary", col("salary") * 100)
display(df2)

In [0]:
from pyspark.sql.functions import expr

#Create a Column from an Existing
df2 = df.withColumn("bonus", expr("CASE WHEN salary = 3000 THEN salary * 0.10 WHEN salary = 4000 THEN salary * 0.08 ELSE salary * 0.20 END"))
display(df2)

In [0]:

df.withColumnRenamed("gender","sex") \
  .show(truncate=False) 

In [0]:
df.drop("salary") \
  .show() 

In [0]:

# Imports
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField 
from pyspark.sql.types import StringType, IntegerType, ArrayType

# Create SparkSession object
spark = SparkSession.builder.appName('SparkByExamples.com').getOrCreate()

# Create data
data = [
    (("James","","Smith"),["Java","Scala","C++"],"OH","M"),
    (("Anna","Rose",""),["Spark","Java","C++"],"NY","F"),
    (("Julia","","Williams"),["CSharp","VB"],"OH","F"),
    (("Maria","Anne","Jones"),["CSharp","VB"],"NY","M"),
    (("Jen","Mary","Brown"),["CSharp","VB"],"NY","M"),
    (("Mike","Mary","Williams"),["Python","VB"],"OH","M")
 ]

# Create schema        
schema = StructType([
     StructField('name', StructType([
        StructField('firstname', StringType(), True),
        StructField('middlename', StringType(), True),
         StructField('lastname', StringType(), True)
     ])),
     StructField('languages', ArrayType(StringType()), True),
     StructField('state', StringType(), True),
     StructField('gender', StringType(), True)
 ])

# Create dataframe
df = spark.createDataFrame(data = data, schema = schema)
df.printSchema()
df.show(truncate=False)

In [0]:
df.filter(df.state == "OH").show(truncate=False)
# Not equals condition
df.filter(df.state != "OH") \
    .show(truncate=False) 
# Another expression
df.filter(~(df.state == "OH")) \
    .show(truncate=False)

df.filter("gender == 'M'").show(truncate=False) # sql 
# For not equal
df.filter("gender != 'M'").show()
df.filter("gender <> 'M'").show() # sql 
df.filter(~(df.gender == "M")).show(truncate=False)

# PySpark Filter with Multiple Conditions
df.filter( (df.state  == "OH") & (df.gender  == "M") ) \
    .show(truncate=False)  




In [0]:
# Filter array elements
from pyspark.sql.functions import col,array_contains
# Filter array elements
df.filter(array_contains(df.languages,"Java")).show(truncate=False) # array contains Java


# # Filter IS IN List values
# li=["OH","NY","DE"]
# df.filter(df.state.isin(li)).show(truncate=False)


# # Filter NOT IS IN List values
# # These show all records with NY (NY is not part of the list)
# li=["OH","AB","DE"]
# df.filter(~df.state.isin(li)).show()
# df.filter(df.state.isin(li)==False).show()

In [0]:

# Using startswith
df.filter(df.state.startswith("N")).show()

# Output
#+--------------------+------------------+-----+------+
#|                name|         languages|state|gender|
#+--------------------+------------------+-----+------+
#|      [Anna, Rose, ]|[Spark, Java, C++]|   NY|     F|
#|[Maria, Anne, Jones]|      [CSharp, VB]|   NY|     M|
#|  [Jen, Mary, Brown]|      [CSharp, VB]|   NY|     M|
#+--------------------+------------------+-----+------+

#using endswith
df.filter(df.state.endswith("H")).show()

#contains
df.filter(df.state.contains("H")).show()

In [0]:

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr
spark = SparkSession.builder.appName('SparkByExamples.com').getOrCreate()

data = [("James", "Sales", 3000), \
    ("Michael", "Sales", 4600), \
    ("Robert", "Sales", 4100), \
    ("Maria", "Finance", 3000), \
    ("James", "Sales", 3000), \
    ("Scott", "Finance", 3300), \
    ("Jen", "Finance", 3900), \
    ("Jeff", "Marketing", 3000), \
    ("Kumar", "Marketing", 2000), \
    ("Saif", "Sales", 4100) \
  ]
columns= ["employee_name", "department", "salary"]
df = spark.createDataFrame(data = data, schema = columns)
df.printSchema()
df.show(truncate=False)

#Distinct
distinctDF = df.distinct()
print("Distinct count: "+str(distinctDF.count()))
distinctDF.show(truncate=False)




In [0]:
#Drop duplicates
df2 = df.dropDuplicates()         # same as  distinct 
print("Distinct count: "+str(df2.count()))
df2.show(truncate=False)



In [0]:
#Drop duplicates on selected columns
dropDisDF = df.dropDuplicates(["department","salary"])
print("Distinct count of department salary : "+str(dropDisDF.count()))
dropDisDF.show(truncate=False)

In [0]:
# Imports
import pyspark
from pyspark.sql import SparkSession

simpleData = [("James","Sales","NY",90000,34,10000), \
    ("Michael","Sales","NY",86000,56,20000), \
    ("Robert","Sales","CA",81000,30,23000), \
    ("Maria","Finance","CA",90000,24,23000), \
    ("Raman","Finance","CA",99000,40,24000), \
    ("Scott","Finance","NY",83000,36,19000), \
    ("Jen","Finance","NY",79000,53,15000), \
    ("Jeff","Marketing","CA",80000,25,18000), \
    ("Kumar","Marketing","NY",91000,50,21000) \
  ]
columns= ["employee_name","department","state","salary","age","bonus"]
# Create SparkSession

df = spark.createDataFrame(data = simpleData, schema = columns)
df.printSchema()
df.show(truncate=False)

In [0]:
# syntax : df.sort("column1", "column2", ascending=[True, False]) 

from pyspark.sql.functions import col,desc,asc

df.sort("salary") .show() # by default ascending
df.sort("salary",ascending=False) .show()
df.sort("salary","department",ascending=False).show()
df.sort(asc("salary"),desc("department")).show() # easy way to multiple
df.sort(col("salary").desc()) .show(truncate=False)
#using desc, asc
df.sort(desc("salary")) .show(truncate=False)
df.sort(asc("salary")) .show(truncate=False)
#using dataframe
df.sort(df.salary.desc()) .show(truncate=False)


# sort by multiple columns 
from pyspark.sql.functions import col,desc,asc
df.sort(col("department").desc(),col("salary").asc()) .show(truncate=False)




In [0]:
# Sorting DataFrame using orderBy()

from pyspark.sql.functions import col, desc, asc

df.orderBy(desc("department"), asc("salary")).show(truncate=False)
#df.orderBy(col("salary").desc(),col("department").asc()).show(truncate=False) wrong 

#df.orderBy(col("salary").desc(),col("department").asc()).show(truncate=False)
df.explain()

In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import approx_count_distinct,collect_list
from pyspark.sql.functions import collect_set,sum,avg,max,countDistinct,count
from pyspark.sql.functions import first, last, kurtosis, min, mean, skewness 
from pyspark.sql.functions import stddev, stddev_samp, stddev_pop, sumDistinct
from pyspark.sql.functions import variance,var_samp,  var_pop
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.appName('SparkByExamples.com').getOrCreate()

simpleData = [("James", "Sales", 3000),
    ("Michael", "Sales", 4600),
    ("Robert", "Sales", 4100),
    ("Maria", "Finance", 3000),
    ("James", "Sales", 3000),
    ("Scott", "Finance", 3300),
    ("Jen", "Finance", 3900),
    ("Jeff", "Marketing", 3000),
    ("Kumar", "Marketing", 2000),
    ("Saif", "Sales", 4100)
  ]
schema = StructType([
    StructField("employee_name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True)
])
  
df = spark.createDataFrame(data=simpleData, schema=schema)
df.printSchema()
df.show(truncate=False)



In [0]:
# returns list of elements in the salary column with duplicate records also
df.select(collect_list("salary")).show(truncate=False)
# returns list of elements in the salary column with distinct  records 
df.select(collect_set("salary")).show(truncate=False)

In [0]:


from pyspark.sql.functions import min, max, avg, count, sum, round
df.groupBy("department").agg(
    round(sum("salary"), 2).alias("sum_salary"),
    round(avg("salary"), 2).alias("avg_salary"),
    round(min("salary"), 2).alias("min_salary"),
    round(max("salary"), 2).alias("max_salary")
).show(truncate=False)

In [0]:
df.select(first("salary")).show(truncate=False)
df.select(last("salary")).show(truncate=False)
df.select(kurtosis("salary")).show(truncate=False)

In [0]:
df.select(max("salary").alias("max_sal")).show(truncate=False)
df.select(min("salary").alias("min_sal")).show(truncate=False)
df.select(mean("salary").alias("meanofsal")).show(truncate=False)
df.select(sumDistinct("salary")).show(truncate=False)

In [0]:
simpleData = [
    ("James", "Sales", 3000),
    ("Michael", "Sales", 4600),
    ("Robert", "Sales", 4100),
    ("Maria", "Finance", 3000),
    ("James", "Sales", 3000),
    ("Scott", "Finance", 3300),
    ("Jen", "Finance", 3900),
    ("Jeff", "Marketing", 3000),
    ("Kumar", "Marketing", 2000),
    ("Saif", "Sales", 4100)
]

columns = ["employee_name", "department", "salary"]
df = spark.createDataFrame(data=simpleData, schema=columns)
df.printSchema()
df.show(truncate=False)

In [0]:
# row_number() example
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

#defines the partitions and order of the dataframe
windowSpec = Window.partitionBy("department").orderBy("salary")

df1 = df.withColumn("row_number", row_number().over(windowSpec))
display(df1)

#df2 = df1.select("employee_name", "department", "salary", "row_number")
#display(df2)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

windowSpec = Window.partitionBy("department").orderBy("salary")
df3 = df1.withColumn("row_number", row_number().over(windowSpec)).filter(col("row_number") == 1)

display(df3)

df4=df1.filter(col("row_number") == 1)
display(df4)


In [0]:
# df1.filter(df1.row_number == 1).show(truncate=False)
# df1.filter((df1.row_number == 1) | (df1.row_number == 2)).show(truncate=False)
display(df1.filter(df1.row_number == 1 | 2))# insted of 1,2 3 will come 

In [0]:
# rank() example
from pyspark.sql.functions import rank
windowSpe = Window.partitionBy("department").orderBy(col("salary").desc(),col("department").desc())
df.withColumn("rank", rank().over(windowSpe)) \
    .show()

In [0]:
from pyspark.sql.functions import dense_rank
windowSpec = Window.partitionBy("department").orderBy("salary")
df.withColumn("dense_rank",dense_rank().over(windowSpec)) \
    .show()

In [0]:
from pyspark.sql.functions import percent_rank
windowSpec = Window.partitionBy("department").orderBy(col("salary").desc())
df.withColumn("percent_rank",percent_rank().over(windowSpec)) \
    .show()

In [0]:
#create the buckets based on count of particular departrment and divide them among two groups called ntile
from pyspark.sql.functions import ntile
windowSpec = Window.partitionBy("department").orderBy("salary")
df.withColumn("ntile",ntile(2).over(windowSpec)) \
    .show()

In [0]:

from pyspark.sql.functions import lag    
df.withColumn("lag",lag("salary",1).over(windowSpec)) \
      .show()

from pyspark.sql.functions import lead    
df.withColumn("lead",lead("salary",1).over(windowSpec)) \
    .show()

In [0]:
windowSpecAgg  = Window.partitionBy("department")
from pyspark.sql.functions import col,avg,sum,min,max,row_number 
df.withColumn("row",row_number().over(windowSpec)) \
  .withColumn("avg", avg(col("salary")).over(windowSpecAgg)) \
  .withColumn("sum", sum(col("salary")).over(windowSpecAgg)) \
  .withColumn("min", min(col("salary")).over(windowSpecAgg)) \
  .withColumn("max", max(col("salary")).over(windowSpecAgg)) \
  .show()




In [0]:
# where row = 1
df.withColumn("row",row_number().over(windowSpec)) \
  .withColumn("avg", avg(col("salary")).over(windowSpecAgg)) \
  .withColumn("sum", sum(col("salary")).over(windowSpecAgg)) \
  .withColumn("min", min(col("salary")).over(windowSpecAgg)) \
  .withColumn("max", max(col("salary")).over(windowSpecAgg)) \
  .where(col("row")==1).select("department","avg","sum","min","max") \
  .show()

In [0]:
# DBTITLE 1,Create and Display DataFrame with Complex Types
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, IntegerType
from pyspark.sql import functions as F

# Sample data
rows = [
    Row(
        id=1,
        names=["Alice", "Bob"],
        scores=[85, 90],
        info=Row(age=25, city="New York"),
        json_str='{"dept": "Engineering", "level": 2}'
    ),
    Row(
        id=2,
        names=["Carol"],
        scores=[78, 88, 92],
        info=Row(age=30, city="San Francisco"),
        json_str='{"dept": "HR", "level": 1}'
    )
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("names", ArrayType(StringType()), True),
    StructField("scores", ArrayType(IntegerType()), True),
    StructField("info", StructType([StructField("age", IntegerType(), True),StructField("city", StringType(), True)]), True),
    StructField("json_str", StringType(), True)
])

df = spark.createDataFrame(rows, schema)
df.createOrReplaceTempView("df")
display(df)


In [0]:
# DBTITLE 1,PySpark - Array Operations
from pyspark.sql import functions as F
#here we have f.size,f.coloum insted of importing all individual functions we directly import all and from F 
# we can use the functions directly by specifying f.size,f.coloums
# Get the length of the 'names' array
array_length_df = df.withColumn("names_length", F.size(F.col("names")))
display(array_length_df)



In [0]:
# Check if 'names' array contains 'Alice'
array_contains_df = array_length_df.withColumn("has_Alice", F.array_contains(F.col("names"), "Alice"))

display(array_contains_df)



Split()

split() sql function returns an array type after splitting the string column by delimiter. Below example split the name column by comma delimiter.

In [0]:
# Split a string column into an array (for demonstration, create a new column)
df_array = array_contains_df.withColumn("city_words", F.split(F.col("info.city"), " "))
display(df_array)

#select only required columns 
display(df_array.select("id", "names", "names_length", "has_Alice", "city_words"))

In [0]:
%sql
select *,size(names) names_length, array_contains(names , 'Alice') has_alice,
split(info.city,' ') city_words   from  df

In [0]:
#accesing field in the info column using . and specifing key  name 
#display(df_array.select("id", "names", "names_length", "has_Alice", "city_words", "info.age","info.city"))

# or we include them in dataframe using with column
df_array_contains = df_array.withColumn("age", F.col("info.age")).withColumn("city", F.col("info.city"))
display(df_array_contains)

PySpark JSON Functions()

from_json()	Converts JSON string into Struct type or Map type.
to_json()	Converts MapType or Struct type to JSON string.
json_tuple()	Extract the Data from JSON and create them as a new columns.
get_json_object()	Extracts JSON element from a JSON string based on json path specified.
schema_of_json()	Create schema string from JSON string

In [0]:
#from_json()	Converts JSON string into Struct type or Map type.
#Create DataFrame with Column containing JSON String
jsonString="""{"Zipcode":704,"ZipCodeType":"STANDARD","City":"PARC PARQUE","State":"PR"}"""
df=spark.createDataFrame([(1, jsonString)],["id","value"])
df.show(truncate=False)

#Convert JSON string column to Map type
from pyspark.sql.types import MapType,StringType
from pyspark.sql.functions import from_json
df2=df.withColumn("value",from_json(df.value,MapType(StringType(),StringType())))
df2.printSchema()
df2.show(truncate=False)
display(df2)
df3=df.withColumn("value",from_json(df.value,MapType(StringType(),StringType())))
display(df3)

In [0]:
#to_json() function is used to convert DataFrame columns MapType or Struct type to JSON string. Here, I am using df2 that created from above from_json() example.
from pyspark.sql.functions import to_json, col
df2_json = df2.withColumn("value_json", to_json(col("value")))
display(df2_json)
# here value : struct type ---> value_json : json type

In [0]:
#Function json_tuple() is used the query or extract the elements from JSON column and create the result as a new columns.
from pyspark.sql.functions import json_tuple, col
df4 = df.select(col("id"), json_tuple(col("value"), "Zipcode", "ZipCodeType", "City", "State")) \
    .toDF("id", "Zipcode_name", "ZipCodeType_name", "City_name", "state_name")
display(df4)
# we use toDF to create a datafram using existing dataframe

In [0]:
#get_json_object() is used to extract the JSON string based on path from the JSON column.
from pyspark.sql.functions import get_json_object
df5 = df.select(
    "id",
    get_json_object("value", "$.ZipCodeType").alias("ZipCodeType"),
    get_json_object("value", "$.City").alias("City"),
    get_json_object("value", "$.State").alias("state")
)
display(df5)

In [0]:
data = [
 ("James,,Smith",["Java","Scala","C++"],["Spark","Java"],"OH","CA"),
 ("Michael,Rose,",["Spark","Java","C++"],["Spark","Java"],"NY","NJ"),
 ("Robert,,Williams",["CSharp","VB"],["Spark","Python"],"UT","NV")
]

from pyspark.sql.types import StringType, ArrayType,StructType,StructField
schema = StructType([ 
    StructField("name",StringType(),True), 
    StructField("languagesAtSchool",ArrayType(StringType()),True), 
    StructField("languagesAtWork",ArrayType(StringType()),True), 
    StructField("currentState", StringType(), True), 
    StructField("previousState", StringType(), True)
  ])

df = spark.createDataFrame(data=data,schema=schema)
df.printSchema()
df.show()

In [0]:
from pyspark.sql import functions as F
display(df)
# 1. Get array length
df_length = df.withColumn("school_length", F.size(F.col("languagesAtSchool")))
display(df_length)

# 2. Check if array contains a value
df_contains = df_length.withColumn("has_Java", F.array_contains(F.col("languagesAtSchool"), "Java"))
display(df_contains)

# 3. Explode array to rows
df_exploded = df_contains.withColumn("exploded_language", F.explode(F.col("languagesAtSchool")))
display(df_exploded)

# 4. Split string column to array
df_split = df_contains.withColumn("nameAsArray", F.split(F.col("name"), ","))
display(df_split)

# 5. Merge columns into array
df_array = df_contains.withColumn("state_array", F.array(F.col("currentState"), F.col("previousState")))
display(df_array)

# 6. Array distinct
df_distinct = df_contains.withColumn("distinct_school", F.array_distinct(F.col("languagesAtSchool")))
display(df_distinct)

# 7. Array union
df_union = df_contains.withColumn("school_union_work", F.array_union(F.col("languagesAtSchool"), F.col("languagesAtWork")))
display(df_union)

# 8. Array intersection
df_intersect = df_contains.withColumn("school_intersect_work", F.array_intersect(F.col("languagesAtSchool"), F.col("languagesAtWork")))
display(df_intersect)

# 9. Array except
df_except = df_contains.withColumn("school_except_work", F.array_except(F.col("languagesAtSchool"), F.col("languagesAtWork")))
display(df_except)

# 10. Array position
df_position = df_contains.withColumn("java_position", F.array_position(F.col("languagesAtSchool"), "Java"))
display(df_position)

# 11. Array reverse
df_reverse = df_contains.withColumn("school_reverse", F.reverse(F.col("languagesAtSchool")))
display(df_reverse)

# 12. Array sort
df_sort = df_contains.withColumn("school_sorted", F.array_sort(F.col("languagesAtSchool")))
display(df_sort)

In [0]:
# Use explode() function to create a new row for each element in the given array column. There are various PySpark SQL explode functions available to work with Array columns.

from pyspark.sql.functions import explode
display(df.select("name", explode("languagesAtSchool")))
df2 = df.withColumn("exploded_language", explode("languagesAtSchool"))
display(df2)

In [0]:
#split() sql function returns an array type after splitting the string column by delimiter. Below example split the name column by comma delimiter.

from pyspark.sql.functions import split
df1 = df.select("name", split(df.name, ",").alias("nameAsArray"))

display(df1)

In [0]:
#Use array() function to create a new array column by merging the data from multiple columns. All input columns must have the same data type.
from pyspark.sql.functions import array
df3=df.select(df.name,array(df.currentState,df.previousState).alias("state"))
display(df3)

In [0]:
# array_contains() sql function is used to check if array column contains a value. Returns null if the array is null, true if the array contains the value, and false otherwise
from pyspark.sql.functions import array_contains
df4 = df.select(df.name,array_contains(df.languagesAtSchool,"Java").alias("array_contains"))
df5 = df.withColumn("array_contains", array_contains(df.languagesAtSchool, "Java"))
display(df4)
display(df5)
